# 02a – Player Team Dashboard: Relación de métricas con Victoria/Impacto

Este cuaderno analiza el dashboard de jugadores por equipo para relacionar las métricas numéricas originales con los indicadores de victoria/derrota (`W`, `L`, `W_PCT`) y el impacto (`PLUS_MINUS`).

**Objetivos principales:**
- Explorar de forma sistemática cómo se comportan las métricas disponibles respecto a los objetivos de resultado e impacto.
- Detectar patrones generales, diferencias Home/Away y posibles incoherencias en columnas de rangos.
- Registrar hallazgos clave para orientar análisis predictivos posteriores sin crear métricas derivadas por jugador.

**Notas:**
- Solo se emplean las columnas originales; se permiten agregaciones para comparaciones (por ejemplo, medias por `GROUP_SET`).
- Las secciones incluyen comentarios que explican qué observar en cada visualización o tabla resultante.


## 1. Configuración e importaciones

Definimos las variables de configuración, importamos las librerías necesarias y establecemos estilos de visualización coherentes. Ajusta `CSV_PATH` con la ruta del archivo CSV a analizar antes de ejecutar el cuaderno.


In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Variables de configuración (ajusta CSV_PATH según tu entorno)
CSV_PATH = Path("../ruta_al_csv.csv")  # ← Actualiza esta ruta al archivo 'player dashboard by team'
SAVE_FIG = True
FIG_DPI = 110
SEED = 42

np.random.seed(SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)
plt.style.use("seaborn-v0_8")
sns.set_context("talk")
warnings.filterwarnings("ignore", category=FutureWarning)

OUTPUT_DIR = Path("outputs")
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

# Utilidades auxiliares

def ensure_dir(path: Path) -> Path:
    """Crea el directorio si no existe y devuelve la ruta."""
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_filename(name: str) -> str:
    """Genera nombres de archivo sencillos reemplazando caracteres problemáticos."""
    keep = [c if c.isalnum() or c in ("_", "-", " ") else "_" for c in name]
    cleaned = "".join(keep).strip().replace(" ", "_")
    return cleaned.lower()


def save_figure(fig: plt.Figure, path: Path) -> None:
    """Guarda una figura si SAVE_FIG es True."""
    if not SAVE_FIG:
        return
    ensure_dir(path.parent)
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")


def compute_spearman_correlations(df: pd.DataFrame, target: str, columns: list[str]) -> pd.DataFrame:
    """Calcula correlaciones de Spearman para las columnas numéricas respecto al objetivo indicado."""
    records = []
    for col in columns:
        if col == target:
            continue
        if col not in df.columns:
            continue
        series = df[[col, target]].dropna()
        if series.shape[0] < 5:
            continue
        if series[col].nunique(dropna=True) < 2 or series[target].nunique(dropna=True) < 2:
            continue
        rho = series[col].corr(series[target], method="spearman")
        if pd.notna(rho):
            records.append({"variable": col, "rho_spearman": rho, "n_observaciones": int(series.shape[0])})
    if not records:
        return pd.DataFrame(columns=["variable", "rho_spearman", "n_observaciones"])
    result = pd.DataFrame(records).sort_values("rho_spearman", ascending=False).reset_index(drop=True)
    return result


# Creación inicial de carpetas de salida
ensure_dir(FIGURES_DIR)
ensure_dir(TABLES_DIR)


## 2. Carga, limpieza ligera y tipado de columnas

Se carga el CSV indicado, se normalizan los nombres de columnas y se reconcilian duplicidades (`TEAM_ID`/`team_id`, `SEASON_YEAR`/`season`). Además, se definen los grupos de columnas categóricas, numéricas y de objetivos, convirtiendo las numéricas a formato adecuado. Finalmente se revisa la estructura general y los valores nulos más frecuentes.


In [ ]:
if not CSV_PATH.exists():
    warnings.warn(f"El archivo no existe en la ruta indicada: {CSV_PATH}")

df = pd.DataFrame()
if CSV_PATH.exists():
    df = pd.read_csv(CSV_PATH)
    df.columns = [c.strip() for c in df.columns]

    if "TEAM_ID" in df.columns and "team_id" in df.columns:
        df["TEAM_ID"] = df["TEAM_ID"].fillna(df["team_id"])
        df["team_id"] = df["team_id"].fillna(df["TEAM_ID"])
    elif "team_id" in df.columns and "TEAM_ID" not in df.columns:
        df["TEAM_ID"] = df["team_id"]
    elif "TEAM_ID" in df.columns and "team_id" not in df.columns:
        df["team_id"] = df["TEAM_ID"]

    if "season" in df.columns and "SEASON_YEAR" in df.columns:
        df["season"] = df["season"].fillna(df["SEASON_YEAR"])
        df["SEASON_YEAR"] = df["SEASON_YEAR"].fillna(df["season"])
    elif "SEASON_YEAR" in df.columns and "season" not in df.columns:
        df["season"] = df["SEASON_YEAR"]
    elif "season" in df.columns and "SEASON_YEAR" not in df.columns:
        df["SEASON_YEAR"] = df["season"]

    if "dataset" in df.columns:
        original_shape = df.shape
        df = df[df["dataset"] == 1].copy()
        if df.shape != original_shape:
            print(f"Filtrado dataset==1: {original_shape} -> {df.shape}")

cat_base = [
    "TEAM_ID",
    "TEAM_NAME",
    "PLAYER_ID",
    "PLAYER_NAME",
    "NICKNAME",
    "GROUP_SET",
    "season",
    "SEASON_YEAR",
    "season_type",
    "endpoint",
]
cat_cols = [c for c in cat_base if c in df.columns]

rank_cols = [c for c in df.columns if c.endswith("_RANK")]
target_cols = [c for c in ["W", "L", "W_PCT", "PLUS_MINUS"] if c in df.columns]

num_cols = [c for c in df.columns if c not in cat_cols]

for col in num_cols:
    if df.empty:
        continue
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Shape del DataFrame: {df.shape}")
if not df.empty:
    display(df.head())
    na_summary = df.isna().sum().sort_values(ascending=False)
    display(na_summary.head(20).to_frame(name="n_nulos"))
    print(f"Columnas categóricas: {len(cat_cols)} | Numéricas: {len(num_cols)} | Objetivos: {target_cols}")


## 3. Correlaciones globales con W_PCT y PLUS_MINUS

Calculamos correlaciones de Spearman entre cada métrica numérica y los objetivos principales. Las tablas ordenadas resaltan asociaciones fuertes; los gráficos de barras muestran los 20 valores más altos y más bajos por objetivo; y los mapas de calor permiten observar agrupaciones de métricas relacionadas con el rendimiento.


In [ ]:
corr_tables = {}

if df.empty:
    warnings.warn("No hay datos cargados; se omiten correlaciones.")
else:
    corr_dir = ensure_dir(FIGURES_DIR / "corr")
    corr_table_dir = ensure_dir(TABLES_DIR / "corr")
    for target in target_cols:
        corr_df = compute_spearman_correlations(df, target, [c for c in num_cols if c in df.columns])
        if corr_df.empty:
            warnings.warn(f"Sin correlaciones válidas para {target}.")
            continue
        corr_tables[target] = corr_df
        csv_path = corr_table_dir / f"corr_spearman_{safe_filename(target)}.csv"
        corr_df.to_csv(csv_path, index=False)
        display(corr_df.head(10))

        top_n = corr_df.head(20)
        bottom_n = corr_df.tail(20)
        for subset, label in [(top_n, "top"), (bottom_n, "bottom")]:
            if subset.empty:
                continue
            fig, ax = plt.subplots(figsize=(10, max(6, 0.35 * len(subset))))
            sns.barplot(
                data=subset,
                y="variable",
                x="rho_spearman",
                palette="crest" if label == "top" else "flare",
                ax=ax,
            )
            ax.set_title(f"Correlaciones {label} con {target}")
            ax.set_xlabel("Rho de Spearman")
            ax.set_ylabel("Métrica")
            ax.axvline(0, color="gray", linestyle="--", linewidth=1)
            plt.tight_layout()
            save_figure(fig, corr_dir / f"{safe_filename(target)}_{safe_filename(label)}_20.png")
            plt.close(fig)

        top_abs = corr_df.reindex(corr_df["rho_spearman"].abs().sort_values(ascending=False).index).head(25)
        if not top_abs.empty:
            fig, ax = plt.subplots(figsize=(12, max(6, 0.4 * len(top_abs))))
            sns.heatmap(
                top_abs.set_index("variable")["rho_spearman"].to_frame(),
                annot=True,
                fmt=".2f",
                cmap="coolwarm",
                center=0,
                ax=ax,
            )
            ax.set_title(f"Top 25 correlaciones (|rho|) con {target}")
            ax.set_xlabel("Rho de Spearman")
            plt.tight_layout()
            save_figure(fig, corr_dir / f"{safe_filename(target)}_heatmap_top25.png")
            plt.close(fig)


## 4. Relaciones directas: dispersión y densidad

Visualizamos cada métrica numérica frente a los objetivos (`W_PCT` y `PLUS_MINUS`). Los gráficos permiten detectar relaciones lineales o no lineales, además de dispersiones altas o grupos particulares. Se emplea `hexbin` cuando la cantidad de observaciones es elevada para mejorar la legibilidad.


In [ ]:
def plot_metric_vs_target(metric: str, target: str) -> None:
    if metric not in df.columns or target not in df.columns or df.empty:
        return
    data = df[[metric, target]].dropna()
    if data.shape[0] < 15 or data[metric].nunique() <= 3 or data[target].nunique() <= 3:
        return
    fig, ax = plt.subplots(figsize=(8, 6))
    if data.shape[0] > 500:
        hb = ax.hexbin(data[metric], data[target], gridsize=30, cmap="viridis", mincnt=1)
        cb = fig.colorbar(hb, ax=ax)
        cb.set_label("Densidad de observaciones")
    else:
        ax.scatter(data[metric], data[target], alpha=0.6, edgecolor="none", s=35, color="#1f77b4")
    ax.set_title(f"{metric} vs {target}")
    ax.set_xlabel(metric)
    ax.set_ylabel(target)
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    save_figure(fig, ensure_dir(FIGURES_DIR / "scatter") / f"{safe_filename(metric)}_vs_{safe_filename(target)}.png")
    plt.close(fig)

if not df.empty and target_cols:
    processed = 0
    for metric in [c for c in num_cols if c in df.columns]:
        if metric in target_cols:
            continue
        for target in target_cols:
            plot_metric_vs_target(metric, target)
        processed += 1
        if processed % 10 == 0:
            print(f"Procesadas {processed} métricas para gráficos de dispersión/hexbin...")


## 5. Comparativa Home vs Away (GROUP_SET)

Si el dataset diferencia actuaciones en casa y fuera mediante `GROUP_SET`, revisamos la distribución de métricas numéricas por grupo, las medias y diferencias, y algunos pares clave con color según el grupo. Esto permite identificar métricas sensibles a la localía.


In [ ]:
if not df.empty and "GROUP_SET" in df.columns and df["GROUP_SET"].nunique() > 1:
    group_dir = ensure_dir(FIGURES_DIR / "group_set")
    violin_dir = ensure_dir(group_dir / "violin")
    delta_fig_dir = ensure_dir(group_dir / "deltas")
    tables_group_dir = ensure_dir(TABLES_DIR / "group_set")

    for metric in [c for c in num_cols if c in df.columns and df[c].nunique() > 5]:
        data = df[[metric, "GROUP_SET"]].dropna()
        if data.empty:
            continue
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.violinplot(data=data, x="GROUP_SET", y=metric, inner="quart", palette="Set2", ax=ax)
        ax.set_title(f"Distribución de {metric} por GROUP_SET")
        ax.set_xlabel("Grupo")
        ax.set_ylabel(metric)
        plt.tight_layout()
        save_figure(fig, violin_dir / f"{safe_filename(metric)}.png")
        plt.close(fig)

    kde_targets = [t for t in ["W_PCT", "PLUS_MINUS"] if t in df.columns]
    for target in kde_targets:
        data = df[[target, "GROUP_SET"]].dropna()
        if data.empty:
            continue
        fig, ax = plt.subplots(figsize=(8, 5))
        for grp, grp_df in data.groupby("GROUP_SET"):
            sns.kdeplot(grp_df[target], label=grp, ax=ax, fill=True, alpha=0.3)
        ax.set_title(f"Distribución de {target} por GROUP_SET")
        ax.set_xlabel(target)
        ax.set_ylabel("Densidad")
        ax.legend(title="Grupo")
        plt.tight_layout()
        save_figure(fig, group_dir / f"kde_{safe_filename(target)}.png")
        plt.close(fig)

    pivot_metrics = [c for c in num_cols if c in df.columns]
    mean_table = df.groupby("GROUP_SET")[pivot_metrics].mean(numeric_only=True).T
    if not mean_table.empty and {"Home", "Away"}.issubset(mean_table.columns):
        mean_table["delta_home_away"] = mean_table.get("Home", np.nan) - mean_table.get("Away", np.nan)
    mean_table = mean_table.sort_values("delta_home_away", ascending=False)
    if not mean_table.empty:
        mean_table.to_csv(tables_group_dir / "home_away_deltas.csv")
        display(mean_table.head(10))

    if "delta_home_away" in mean_table.columns:
        top20 = mean_table.head(20)
        bottom20 = mean_table.tail(20)
        for subset, label in [(top20, "top"), (bottom20, "bottom")]:
            if subset.empty:
                continue
            fig, ax = plt.subplots(figsize=(9, max(6, 0.4 * len(subset))))
            sns.barplot(
                x="delta_home_away",
                y=subset.index,
                data=subset.reset_index(),
                palette="crest" if label == "top" else "rocket",
                ax=ax,
            )
            ax.set_title(f"Diferencia Home-Away ({label})")
            ax.set_xlabel("Home - Away")
            ax.set_ylabel("Métrica")
            ax.axvline(0, color="gray", linestyle="--", linewidth=1)
            plt.tight_layout()
            save_figure(fig, delta_fig_dir / f"home_away_delta_{label}_20.png")
            plt.close(fig)

    scatter_pairs = [
        ("FG_PCT", "W_PCT"),
        ("FG3_PCT", "W_PCT"),
        ("AST", "W_PCT"),
        ("TOV", "W_PCT"),
        ("REB", "W_PCT"),
        ("PTS", "W_PCT"),
        ("FG_PCT", "PLUS_MINUS"),
        ("FG3_PCT", "PLUS_MINUS"),
        ("AST", "PLUS_MINUS"),
        ("TOV", "PLUS_MINUS"),
        ("REB", "PLUS_MINUS"),
        ("PTS", "PLUS_MINUS"),
    ]
    for x, y in scatter_pairs:
        if x not in df.columns or y not in df.columns:
            continue
        data = df[[x, y, "GROUP_SET"]].dropna()
        if data.empty or data[x].nunique() < 4 or data[y].nunique() < 4:
            continue
        fig, ax = plt.subplots(figsize=(7, 6))
        sns.scatterplot(data=data, x=x, y=y, hue="GROUP_SET", alpha=0.7, ax=ax)
        ax.set_title(f"{x} vs {y} por GROUP_SET")
        ax.set_xlabel(x)
        ax.set_ylabel(y)
        ax.legend(title="Grupo")
        plt.tight_layout()
        save_figure(fig, group_dir / f"scatter_{safe_filename(x)}_vs_{safe_filename(y)}.png")
        plt.close(fig)
else:
    warnings.warn("No se generaron comparativas Home/Away (falta GROUP_SET o datos insuficientes).")


## 6. Coherencia de columnas `_RANK`

Se comprueba si los rangos (`*_RANK`) mantienen coherencia con su métrica base: esperamos correlaciones negativas (a menor rango, mejor valor). Además, se visualizan las distribuciones de `W_PCT` por cuartiles del rango para detectar inconsistencias.


In [ ]:
rank_results = []
if df.empty or not rank_cols:
    warnings.warn("No hay columnas _RANK disponibles para evaluar.")
else:
    ranks_dir = ensure_dir(FIGURES_DIR / "ranks")
    ranks_table_dir = ensure_dir(TABLES_DIR / "ranks")
    for rank_col in rank_cols:
        metric = rank_col.replace("_RANK", "")
        if metric not in df.columns:
            continue
        data = df[[metric, rank_col]].dropna()
        if data.shape[0] < 10 or data[metric].nunique() < 3 or data[rank_col].nunique() < 3:
            continue
        rho = data[metric].corr(data[rank_col], method="spearman")
        rank_results.append({
            "rank_col": rank_col,
            "metric": metric,
            "rho_spearman": rho,
            "n_observaciones": int(data.shape[0]),
        })

        fig, ax = plt.subplots(figsize=(6, 5))
        sns.scatterplot(data=data, x=metric, y=rank_col, alpha=0.7, ax=ax)
        ax.set_title(f"Coherencia {metric} vs {rank_col} (rho={rho:.2f})")
        ax.set_xlabel(metric)
        ax.set_ylabel(rank_col)
        plt.tight_layout()
        save_figure(fig, ranks_dir / f"scatter_{safe_filename(metric)}_{safe_filename(rank_col)}.png")
        plt.close(fig)

        if "W_PCT" in df.columns:
            temp = df[[rank_col, "W_PCT"]].dropna()
            if not temp.empty:
                quartiles = pd.qcut(temp[rank_col], q=4, labels=["Q1", "Q2", "Q3", "Q4"], duplicates="drop")
                fig, ax = plt.subplots(figsize=(6, 5))
                sns.boxplot(x=quartiles, y=temp["W_PCT"], palette="Blues", ax=ax)
                ax.set_title(f"W_PCT por cuartiles de {rank_col}")
                ax.set_xlabel("Cuartil del rango")
                ax.set_ylabel("W_PCT")
                plt.tight_layout()
                save_figure(fig, ranks_dir / f"boxplot_wpct_{safe_filename(rank_col)}.png")
                plt.close(fig)

    if rank_results:
        rank_df = pd.DataFrame(rank_results).sort_values("rho_spearman")
        (ranks_table_dir / "coherencia_ranks.csv").write_text(rank_df.to_csv(index=False))
        display(rank_df.head(10))


## 7. Relación eficiencia vs volumen

Revisamos pares de métricas que combinan porcentajes con volumen (intentos o totales) para identificar zonas de mayor eficacia. Los colores reflejan `W_PCT` o `PLUS_MINUS`, facilitando detectar jugadores con impacto positivo.


In [ ]:
if df.empty:
    warnings.warn("Sin datos para evaluar eficiencia vs volumen.")
else:
    eff_dir = ensure_dir(FIGURES_DIR / "efficiency_volume")
    pairs = [
        ("FG_PCT", "FGA"),
        ("FG3_PCT", "FG3A"),
        ("FT_PCT", "FTA"),
        ("AST", "TOV"),
        ("OREB", "DREB"),
    ]
    color_targets = [t for t in ["W_PCT", "PLUS_MINUS"] if t in df.columns]
    for x, y in pairs:
        if x not in df.columns or y not in df.columns:
            continue
        data = df[[x, y] + color_targets].dropna()
        if data.shape[0] < 20:
            continue
        for target in color_targets:
            fig, ax = plt.subplots(figsize=(7, 6))
            scatter = ax.scatter(
                data[x],
                data[y],
                c=data[target],
                cmap="viridis",
                alpha=0.7,
                edgecolor="none",
            )
            ax.set_title(f"{x} vs {y} coloreado por {target}")
            ax.set_xlabel(x)
            ax.set_ylabel(y)
            cbar = plt.colorbar(scatter, ax=ax)
            cbar.set_label(target)
            ax.grid(True, linestyle="--", alpha=0.4)
            plt.tight_layout()
            save_figure(fig, eff_dir / f"{safe_filename(x)}_vs_{safe_filename(y)}_{safe_filename(target)}.png")
            plt.close(fig)


## 8. Defensa y control de faltas

Analizamos cómo las acciones defensivas (`STL`, `BLK`) y las faltas (`PF`, `BLKA`) se relacionan con los objetivos. Se incluyen diagramas de dispersión/hexbin y un mini mapa de calor con las correlaciones disponibles.


In [ ]:
defense_dir = ensure_dir(FIGURES_DIR / "defense")
def_metrics = ["STL", "BLK", "PF", "BLKA"]
def_targets = [t for t in ["W_PCT", "PLUS_MINUS"] if t in df.columns]

for metric in def_metrics:
    if metric not in df.columns:
        continue
    for target in def_targets:
        plot_metric_vs_target(metric, target)

if def_targets:
    corr_data = []
    for metric in def_metrics:
        if metric not in df.columns:
            continue
        for target in def_targets:
            valid = df[[metric, target]].dropna()
            if valid.shape[0] < 5:
                continue
            rho = valid[metric].corr(valid[target], method="spearman")
            corr_data.append({"Métrica": metric, "Objetivo": target, "rho_spearman": rho})
    if corr_data:
        corr_df = pd.DataFrame(corr_data)
        pivot = corr_df.pivot(index="Métrica", columns="Objetivo", values="rho_spearman")
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.heatmap(pivot, annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
        ax.set_title("Correlaciones defensa/faltas vs objetivos")
        plt.tight_layout()
        save_figure(fig, defense_dir / "correlaciones_defensa.png")
        plt.close(fig)


## 9. Perspectiva por equipo y jugador

Se analizan diferencias entre equipos y jugadores: boxplots de `W_PCT` por equipo, barras de `PLUS_MINUS` por jugador dentro de cada equipo (coloreadas por minutos), y detección de outliers interesantes para profundizar en futuros análisis.


In [ ]:
team_dir = ensure_dir(FIGURES_DIR / "teams")

if "TEAM_NAME" in df.columns and "W_PCT" in df.columns:
    data = df[["TEAM_NAME", "W_PCT"]].dropna()
    if data["TEAM_NAME"].nunique() > 1:
        fig, ax = plt.subplots(figsize=(max(10, data["TEAM_NAME"].nunique() * 0.6), 6))
        sns.boxplot(data=data, x="TEAM_NAME", y="W_PCT", palette="Set3", ax=ax)
        ax.set_title("Distribución de W_PCT por equipo")
        ax.set_xlabel("Equipo")
        ax.set_ylabel("W_PCT")
        ax.tick_params(axis="x", rotation=45, ha="right")
        plt.tight_layout()
        save_figure(fig, team_dir / "boxplot_wpct_por_equipo.png")
        plt.close(fig)

if {"TEAM_NAME", "PLAYER_NAME", "PLUS_MINUS"}.issubset(df.columns):
    pivot_cols = [c for c in ["MIN"] if c in df.columns]
    for team, team_df in df.groupby("TEAM_NAME"):
        team_df = team_df.dropna(subset=["PLAYER_NAME", "PLUS_MINUS"])
        if team_df.empty:
            continue
        team_df = team_df.sort_values("PLUS_MINUS", ascending=False)
        fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(team_df))))
        if "MIN" in pivot_cols and team_df["MIN"].nunique() > 1:
            norm = plt.Normalize(team_df["MIN"].min(), team_df["MIN"].max())
            cmap = plt.cm.viridis
            colors = [cmap(norm(v)) for v in team_df["MIN"]]
            sns.barplot(x="PLUS_MINUS", y="PLAYER_NAME", data=team_df, palette=colors, ax=ax)
            sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
            sm.set_array([])
            cbar = fig.colorbar(sm, ax=ax)
            cbar.set_label("MIN")
        else:
            sns.barplot(x="PLUS_MINUS", y="PLAYER_NAME", data=team_df, palette="Blues", ax=ax)
        ax.set_title(f"PLUS_MINUS por jugador – {team}")
        ax.set_xlabel("PLUS_MINUS")
        ax.set_ylabel("Jugador")
        plt.tight_layout()
        save_figure(fig, team_dir / f"{safe_filename(team)}_players_plusminus.png")
        plt.close(fig)

if "W_PCT" in df.columns and "PTS" in df.columns:
    temp = df[["PLAYER_NAME", "TEAM_NAME", "PTS", "W_PCT"]].dropna()
    if not temp.empty:
        temp = temp.sort_values(["PTS", "W_PCT"], ascending=[False, True]).head(10)
        (TABLES_DIR / "insights_pts_alto_wpct_bajo.csv").write_text(temp.to_csv(index=False))
        display(temp)

if "PLUS_MINUS" in df.columns and "MIN" in df.columns:
    temp = df[["PLAYER_NAME", "TEAM_NAME", "MIN", "PLUS_MINUS"]].dropna()
    if not temp.empty:
        temp = temp.sort_values(["MIN", "PLUS_MINUS"], ascending=[False, True]).head(10)
        (TABLES_DIR / "insights_min_alto_plusminus_bajo.csv").write_text(temp.to_csv(index=False))
        display(temp)


## 10. Fantasía y logros individuales

Se evalúa cómo los puntos de fantasía (`NBA_FANTASY_PTS`, `WNBA_FANTASY_PTS`) y los logros (`DD2`, `TD3`) se asocian con los objetivos. Se incluyen correlaciones y visualizaciones con bins de `W_PCT` para identificar tendencias.


In [ ]:
fantasy_dir = ensure_dir(FIGURES_DIR / "fantasy")
fantasy_tables = []

fantasy_metrics = [c for c in ["NBA_FANTASY_PTS", "WNBA_FANTASY_PTS"] if c in df.columns]
fantasy_targets = [t for t in ["W_PCT", "PLUS_MINUS"] if t in df.columns]

for metric in fantasy_metrics:
    for target in fantasy_targets:
        data = df[[metric, target]].dropna()
        if data.shape[0] < 15:
            continue
        rho = data[metric].corr(data[target], method="spearman")
        fantasy_tables.append({"metric": metric, "objetivo": target, "rho_spearman": rho, "n": int(data.shape[0])})
        fig, ax = plt.subplots(figsize=(7, 6))
        if data.shape[0] > 500:
            hb = ax.hexbin(data[metric], data[target], gridsize=30, cmap="magma", mincnt=1)
            cbar = fig.colorbar(hb, ax=ax)
            cbar.set_label("Densidad")
        else:
            ax.scatter(data[metric], data[target], alpha=0.6, color="#d62728", edgecolor="none")
        ax.set_title(f"{metric} vs {target} (rho={rho:.2f})")
        ax.set_xlabel(metric)
        ax.set_ylabel(target)
        plt.tight_layout()
        save_figure(fig, fantasy_dir / f"{safe_filename(metric)}_vs_{safe_filename(target)}.png")
        plt.close(fig)

for binary in ["DD2", "TD3"]:
    if binary not in df.columns or "W_PCT" not in df.columns:
        continue
    data = df[[binary, "W_PCT"]].dropna()
    if data.empty or data[binary].nunique() <= 1:
        continue
    bins = np.linspace(data["W_PCT"].min(), data["W_PCT"].max(), 6)
    if len(np.unique(bins)) < 2:
        continue
    labels = [f"Bin {i+1}" for i in range(len(bins) - 1)]
    bin_assign = pd.cut(data["W_PCT"], bins=bins, labels=labels, include_lowest=True, duplicates="drop")
    prop = data.groupby(bin_assign)[binary].mean().dropna()
    if not prop.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        sns.barplot(x=prop.index, y=prop.values, palette="crest", ax=ax)
        ax.set_title(f"Proporción de {binary}=1 por tramo de W_PCT")
        ax.set_xlabel("Tramo de W_PCT")
        ax.set_ylabel(f"% {binary}=1")
        ax.set_ylim(0, 1)
        plt.tight_layout()
        save_figure(fig, fantasy_dir / f"proporcion_{safe_filename(binary)}.png")
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.stripplot(x=binary, y="W_PCT", data=data, jitter=True, alpha=0.5, ax=ax)
    ax.set_title(f"{binary} vs W_PCT")
    ax.set_xlabel(binary)
    ax.set_ylabel("W_PCT")
    plt.tight_layout()
    save_figure(fig, fantasy_dir / f"strip_{safe_filename(binary)}.png")
    plt.close(fig)

if fantasy_tables:
    fantasy_df = pd.DataFrame(fantasy_tables)
    ensure_dir(TABLES_DIR / "fantasy").joinpath("fantasy_relations.csv").write_text(fantasy_df.to_csv(index=False))
    display(fantasy_df)


## 11. Multicolinealidad entre predictores

Se inspecciona la correlación entre las métricas numéricas (excluyendo los objetivos) para detectar redundancias. Se genera un mapa de calor completo y otro reducido con las 40 columnas de mayor varianza cuando es necesario.


In [ ]:
multicol_dir = ensure_dir(FIGURES_DIR / "multicollinearity")
if df.empty:
    warnings.warn("No se puede evaluar la multicolinealidad sin datos.")
else:
    predictors = [c for c in num_cols if c in df.columns and c not in target_cols]
    data = df[predictors].dropna(axis=1, how="all")
    if data.empty or data.shape[1] < 2:
        warnings.warn("Columnas numéricas insuficientes para correlaciones.")
    else:
        corr_matrix = data.corr(method="spearman")
        fig, ax = plt.subplots(figsize=(max(12, 0.4 * corr_matrix.shape[1]), max(10, 0.4 * corr_matrix.shape[0])))
        sns.heatmap(corr_matrix, cmap="coolwarm", center=0, ax=ax)
        ax.set_title("Correlación Spearman entre predictores")
        plt.tight_layout()
        save_figure(fig, multicol_dir / "heatmap_completo.png")
        plt.close(fig)

        variances = data.var().sort_values(ascending=False)
        top_cols = variances.head(40).index
        if len(top_cols) > 2 and len(top_cols) < len(predictors):
            corr_top = data[top_cols].corr(method="spearman")
            fig, ax = plt.subplots(figsize=(max(12, 0.4 * len(top_cols)), max(10, 0.4 * len(top_cols))))
            sns.heatmap(corr_top, cmap="coolwarm", center=0, ax=ax)
            ax.set_title("Correlación Spearman (Top 40 varianza)")
            plt.tight_layout()
            save_figure(fig, multicol_dir / "heatmap_top40.png")
            plt.close(fig)


## 12. Resumen ejecutivo

- **Drivers positivos de W_PCT:** _Completar tras la ejecución_ (revisar tablas de correlación y gráficas de eficiencia).
- **Drivers negativos de W_PCT:** _Completar tras la ejecución_ (considerar métricas con correlación negativa y diferencias Home/Away).
- **Variables con gran diferencia Home–Away:** _Completar revisando `home_away_deltas.csv` y los gráficos de barras_.
- **Outliers relevantes:** _Anotar jugadores/equipos detectados en las tablas de insights (`PTS` alto con `W_PCT` bajo, `MIN` alto con `PLUS_MINUS` bajo) y en los gráficos de dispersión_.

> Este espacio sirve para documentar hallazgos clave luego de inspeccionar las visualizaciones y tablas generadas.


## 13. Guardado final y reproducibilidad

Se verifica la creación de carpetas de salida, se contabilizan archivos exportados y se registran versiones de librerías y metadatos básicos de ejecución.


In [ ]:
from datetime import datetime

figure_count = sum(len(files) for _, _, files in os.walk(FIGURES_DIR)) if FIGURES_DIR.exists() else 0
table_count = sum(len(files) for _, _, files in os.walk(TABLES_DIR)) if TABLES_DIR.exists() else 0

print(f"Figuras guardadas: {figure_count}")
print(f"Tablas guardadas: {table_count}")
print(f"Directorios de salida: {FIGURES_DIR.resolve()} | {TABLES_DIR.resolve()}")

print("
Versiones de librerías clave:")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"seaborn: {sns.__version__}")

print(f"Semilla utilizada: {SEED}")
print(f"Fecha/Hora de ejecución: {datetime.now().isoformat(timespec='seconds')}")
